# 16 — Modelos de Defasagens Distribuídas (DLM)

Etapa 2 do TCC. O notebook não refaz Granger. Ele documenta a execução validada do DLM em painel UF × mês e lê as tabelas versionadas em `outputs/tables/dlm`. O HTML final é externo ao GitHub.


## 1. Objetivo
Investigar como a ocorrência de desastres se associa à variação mensal da inadimplência contemporaneamente e até K meses depois, separando efeito contemporâneo, perfil de lags, teste conjunto e efeito acumulado no horizonte K.


## 2. Relação com Granger
Granger testa informação temporal precedente. O DLM estima um perfil de associação distribuída no tempo em painel. Um resultado em um método não contradiz automaticamente o outro.


## 3. Formulação final
Após a validação de estacionariedade, a equação principal é: $\Delta I_{it}=\alpha_i+\lambda_t+\sum_{k=0}^{K}\beta_k\log(1+X_{i,t-k})+\varepsilon_{it}$. A mudança em relação ao PDF, que previa I em nível, está registrada em `docs/dlm_resultados.md`.


In [1]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
OUT=ROOT/'outputs/tables/dlm'
res=pd.read_csv(OUT/'resultados_sintese.csv')
est=pd.read_csv(OUT/'estimabilidade.csv')
st=pd.read_csv(OUT/'estacionariedade_painel.csv')
val=pd.read_csv(OUT/'validacao_atlas.csv')
print({'cenarios_planejados':len(est),'modelos_estimados':len(res),'ufs':27,'K_selecionado':sorted(res.K.unique())})


{'cenarios_planejados': 42, 'modelos_estimados': 40, 'ufs': 27, 'K_selecionado': [3]}


## 4. Estrutura e validação do painel
O painel possui 27 UFs × 144 meses = 3.888 UF-mês, jan/2013–dez/2024, sem duplicidades. A taxa de inadimplência é reconstruída como 100 × carteira inadimplente / carteira ativa. O Atlas bruto foi reconciliado integralmente.


In [2]:
assert val.ok.all()
print('Atlas/painel reconciliados:', int(val.iloc[0].atlas), 'registros 2013–2024')


Atlas/painel reconciliados: 40339 registros 2013–2024


## 5. Construção das defasagens e leakage
`src/dlm.py` ordena por UF e data e aplica `groupby("uf").shift(k)`. Há assert explícito de que o primeiro mês de cada UF não recebe lag da UF anterior.


## 6. Estacionariedade
A série em nível não apresentou evidência suficiente de estacionariedade no painel; ΔI melhorou substancialmente. A transformação não foi escolhida por significância dos DLMs.


In [3]:
display(st)


## 7. Critérios de estimabilidade
A classificação ex ante usa número de eventos, UF-mês positivos, UFs expostas, variação within e concentração. Onda de Calor/Baixa Umidade e barragens não são estimadas no pré-pandemia.


In [4]:
display(est[est.estimabilidade!='estimável'])


## 8. Transformação das contagens
A especificação principal usa `log1p(X)` por assimetria e presença de zeros. Contagem em nível é uma robustez limitada e não é usada para escolher resultados por p-valor.


## 9. Efeitos fixos
Principal: FE de UF + FE mês-ano completo. Assim, β é identificado pela heterogeneidade de exposição entre UFs dentro do mesmo mês, após remover heterogeneidade permanente e choques nacionais comuns.


## 10. Horizonte K
Grade pré-fixada K ∈ {3,6,9,12}, comparada em amostra comum. BIC selecionou K=3 em todos os 40 modelos estimados.


## 11. DLM irrestrito
O modelo irrestrito foi sempre o ponto de partida. Os coeficientes β0…βK, IC95%, teste conjunto e soma dos coeficientes foram calculados com a matriz completa de covariância.


## 12. Multicolinearidade e Almon
Maior correlação absoluta entre lags ≈0,461; condition numbers ficaram muito abaixo de 30. O gatilho predefinido para Almon não foi atingido. Portanto, nenhum resultado final depende de q=2 ou q=3.


## 13. Inferência
Principal: erros-padrão clusterizados por UF, 27 clusters, referência t/F com 26 gl. Robustez: Driscoll–Kraay bandwidth 12. A limitação de apenas 27 clusters permanece explícita.


## 14. Teste conjunto e efeito acumulado
$H_0: \beta_0=...=\beta_K=0$ avalia o perfil completo. O efeito acumulado no horizonte K é $\sum \beta_k$ e não deve ser chamado de efeito de longo prazo quando K é finito.


## 15. Múltiplos testes
BH/FDR é aplicado por período × nível (Total/Grupo/Tipologia), preservando o tamanho planejado da família. Cenários não estimáveis entram implicitamente como p=1. Nenhum lag individual permaneceu significativo após FDR.


## 16. Resultado geral — total de desastres


In [5]:
display(res[res.nivel=='Total'])


## 17. Grupos


In [6]:
display(res[res.nivel=='Grupo'])


## 18. Tipologias e resultados após FDR
O único efeito acumulado que permaneceu significativo após FDR foi Onda de Frio no pré-pandemia. Cinco tipologias do período total também rejeitam o teste conjunto após FDR, mas sem efeito acumulado FDR-significativo.


In [7]:
display(res[(res.q_p_acumulado<.05)|(res.q_p_conjunto<.05)].sort_values(['periodo','exposicao']))


## 19. Onda de Frio no pré-pandemia
Efeito acumulado=0,047964 pp por unidade de log1p(X), IC95% [0,022756; 0,073172], p=0,000589, q=0,009428; teste conjunto q=0,001993. Ressalva: 60 registros, 21 UF-mês positivos e apenas 6 UFs com ocorrência.


## 20. Diagnósticos finais
Na execução completa: 11 modelos “adequados”, 29 “adequados com ressalvas” e nenhum “inadequado para inferência substantiva”. Não houve rejeição de autocorrelação residual nos 40 modelos; dependência transversal residual apareceu em 19, razão para manter DK visível.


## 21. Robustez
A multiverse limitada incluiu contagem × log1p, controles temporais alternativos e K vizinho. Onda de Frio pré manteve sinal positivo em todas as variantes. Não foram criadas especificações adicionais para resgatar significância.


## 22. Comparação com Granger
Granizo, sinal exploratório sensível no fechamento de Granger, não apresenta efeito acumulado FDR-significativo no DLM. O DLM acrescenta um sinal específico de Onda de Frio no pré, sob forte ressalva de esparsidade. As hipóteses dos métodos são diferentes.


## 23. Limitações
Associação condicional não é efeito causal. TWFE não elimina confundidores UF-tempo; 27 clusters limitam aproximações assintóticas; tipologias raras têm baixa informação; os recortes total e pré se sobrepõem e significância diferente não é teste de diferença entre períodos.


## 24. Implicações para SARIMAX
SARIMAX não é executado aqui. A próxima etapa deve preservar a transformação estacionária da inadimplência, usar horizontes parcimoniosos, não selecionar regressoras automaticamente pelos p-valores DLM e enfatizar validação fora da amostra.


In [8]:
print('Validação do notebook: resultados FDR acumulados=',int((res.q_p_acumulado<.05).sum()),'; testes conjuntos FDR=',int((res.q_p_conjunto<.05).sum()))


Validação do notebook: resultados FDR acumulados= 1 ; testes conjuntos FDR= 6
